<a href="https://colab.research.google.com/github/Valbu11/Neurovault-Brain-Data-Analysis/blob/main/notebooks/03_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive montado ✅')

Mounted at /content/drive
Google Drive montado ✅


# 🧠 Notebook 03 — SQL Analysis

**Goal:** Load the clean data into a SQLite database and answer real analytical questions using pure SQL.

**What you will learn:**
- How to create a local SQLite database with Python
- How to load a DataFrame into a SQL table
- How to run SQL queries: SELECT, WHERE, GROUP BY, ORDER BY, JOIN
- How to combine SQL results with Pandas

**Input:** `data/processed/collections_clean.csv` + `data/processed/images_clean.csv`  
**Output:** `data/processed/neurovault.db` (SQLite database)

---
## 1. Import libraries

In [3]:
import pandas as pd
import sqlite3
import os

print('Libraries imported ✅')

Libraries imported ✅


---
## 2. Load clean data

In [4]:
df_c = pd.read_csv('/content/drive/MyDrive/neurovault/processed/collections_clean.csv', low_memory=False)
df_i = pd.read_csv('/content/drive/MyDrive/neurovault/processed/images_clean.csv', low_memory=False)

print(f'Collections: {df_c.shape[0]:,} rows × {df_c.shape[1]} columns')
print(f'Images:      {df_i.shape[0]:,} rows × {df_i.shape[1]} columns')

Collections: 17,216 rows × 18 columns
Images:      2,000 rows × 13 columns


---
## 3. Create SQLite database

SQLite creates a local `.db` file — no server needed, no installation required.
We load both DataFrames as SQL tables inside this database.

In [5]:
DB_PATH = '/content/drive/MyDrive/neurovault/processed/neurovault.db'

conn = sqlite3.connect(DB_PATH)

df_c.to_sql('collections', conn, if_exists='replace', index=False)
df_i.to_sql('images', conn, if_exists='replace', index=False)

print(f'Database created at: {DB_PATH} ✅')
print(f'  Table: collections → {df_c.shape[0]:,} rows')
print(f'  Table: images      → {df_i.shape[0]:,} rows')

# Helper function to run queries easily
def query(sql):
    return pd.read_sql_query(sql, conn)

Database created at: /content/drive/MyDrive/neurovault/processed/neurovault.db ✅
  Table: collections → 17,216 rows
  Table: images      → 2,000 rows


---
## 4. Query 1 — How many studies per year?

**SQL concepts:** SELECT, GROUP BY, ORDER BY

In [6]:
result = query("""
    SELECT
        year,
        COUNT(*) AS total_studies
    FROM collections
    WHERE year IS NOT NULL
    GROUP BY year
    ORDER BY year ASC
""")

print('Studies per year:')
print(result.to_string(index=False))

Studies per year:
 year  total_studies
 2013             11
 2014             84
 2015            334
 2016            484
 2017            618
 2018            913
 2019            971
 2020           2560
 2021           2239
 2022            689
 2023           2197
 2024           2665
 2025           2621
 2026            830


---
## 5. Query 2 — What percentage of studies have a DOI?

**SQL concepts:** ROUND, AVG, calculated columns

In [7]:
result = query("""
    SELECT
        COUNT(*) AS total_studies,
        SUM(has_doi) AS with_doi,
        SUM(1 - has_doi) AS without_doi,
        ROUND(AVG(has_doi) * 100, 1) AS pct_with_doi
    FROM collections
""")

print('DOI coverage:')
print(result.to_string(index=False))

DOI coverage:
 total_studies  with_doi  without_doi  pct_with_doi
         17216       939        16277           5.5


---
## 6. Query 3 — Top 10 most prolific collections by number of images

**SQL concepts:** ORDER BY DESC, LIMIT

In [8]:
result = query("""
    SELECT
        id,
        name,
        number_of_images,
        year,
        has_doi
    FROM collections
    WHERE number_of_images IS NOT NULL
    ORDER BY number_of_images DESC
    LIMIT 10
""")

print('Top 10 collections by number of images:')
print(result.to_string(index=False))

Top 10 collections by number of images:
   id                                                                                                                name  number_of_images  year  has_doi
21705                                                                                                  Parcels_17Networks             65184  2025        0
 9494                      Diffusion-informed spatial smoothing of fMRI data in white matter using spectral graph filters             37145  2021        0
21696                                                                                                   Parcels_7Networks             33256  2025        0
16103                                           Brain topography beyond parcellations: Local gradients of functional maps             29464  2024        1
 4337                                                        The Human Connectome Project: A data acquisition perspective             18070  2018        1
 8996                         

---
## 7. Query 4 — Image modality breakdown

**SQL concepts:** GROUP BY, COUNT, percentage calculation

In [9]:
result = query("""
    SELECT
        modality,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM images), 1) AS percentage
    FROM images
    GROUP BY modality
    ORDER BY total DESC
""")

print('Image modality breakdown:')
print(result.to_string(index=False))

Image modality breakdown:
      modality  total  percentage
     fMRI-BOLD   1582        79.1
       Unknown    388        19.4
Structural MRI     13         0.7
         Other      6         0.3
 Diffusion MRI      5         0.3
      fMRI-CBV      2         0.1
      fMRI-CBF      2         0.1
           MEG      1         0.1
           EEG      1         0.1


---
## 8. Query 5 — Growth rate year over year

**SQL concepts:** Subquery, calculated columns, LAG simulation with self-join

In [10]:
result = query("""
    WITH yearly AS (
        SELECT
            year,
            COUNT(*) AS total
        FROM collections
        WHERE year IS NOT NULL AND year >= 2013
        GROUP BY year
    )
    SELECT
        a.year,
        a.total,
        b.total AS prev_year_total,
        ROUND((a.total - b.total) * 100.0 / b.total, 1) AS growth_pct
    FROM yearly a
    LEFT JOIN yearly b ON a.year = b.year + 1
    ORDER BY a.year
""")

print('Year-over-year growth:')
print(result.to_string(index=False))

Year-over-year growth:
 year  total  prev_year_total  growth_pct
 2013     11              NaN         NaN
 2014     84             11.0       663.6
 2015    334             84.0       297.6
 2016    484            334.0        44.9
 2017    618            484.0        27.7
 2018    913            618.0        47.7
 2019    971            913.0         6.4
 2020   2560            971.0       163.6
 2021   2239           2560.0       -12.5
 2022    689           2239.0       -69.2
 2023   2197            689.0       218.9
 2024   2665           2197.0        21.3
 2025   2621           2665.0        -1.7
 2026    830           2621.0       -68.3


---
## 9. Query 6 — JOIN: images per collection (top 10)

**SQL concepts:** JOIN between two tables

In [11]:
result = query("""
    SELECT
        c.name AS collection_name,
        c.year,
        COUNT(i.id) AS image_count,
        i.modality
    FROM images i
    JOIN collections c ON CAST(i.collection AS TEXT) LIKE '%' || CAST(c.id AS TEXT) || '%'
    GROUP BY c.id, i.modality
    ORDER BY image_count DESC
    LIMIT 10
""")

print('Top collections by image count (with modality):')
print(result.to_string(index=False))

Top collections by image count (with modality):
                                                                                                                                     collection_name  year  image_count  modality
                                                                                     The Neural Basis of Loss Aversion in Decision-Making Under Risk  2013         1139 fMRI-BOLD
                                                                       A Sensitive and Specific Neural Signature for Picture-Induced Negative Affect  2015         1002 fMRI-BOLD
                                                                                                                                                  FF  2013          361 fMRI-BOLD
                                        Escaping the here and now: Evidence for a role of the default mode network in perceptually decoupled thought  2013          144 fMRI-BOLD
                                                              

---
## 10. Close connection

In [12]:
conn.close()
print('Database connection closed ✅')
print()
print('Next step → 04_visualization.ipynb 🚀')

Database connection closed ✅

Next step → 04_visualization.ipynb 🚀


---
## ✅ What we accomplished

- Created a local SQLite database from clean CSV data
- Answered 6 analytical questions using pure SQL
- Used SELECT, WHERE, GROUP BY, ORDER BY, LIMIT, JOIN, CTEs and subqueries
- Combined SQL results with Pandas for easy reading

**Next notebook:** `04_visualization.ipynb` — turn these findings into charts and visual stories.